# TP5 — Couche Silver : normalisation, dépivotage, qualité

Ce notebook lit la couche Bronze (`parse_ok=True`) et alimente :
- **`silver_sensor_reading`** — format long (1 ligne par capteur par relevé), avec flags `is_missing`, `is_duplicate`, `is_outlier`
- **`silver_incident`** — incidents normalisés, liés à `operator` via `operator_id`


## 0. Connexion

In [ ]:
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
import pandas as pd
import numpy as np
from datetime import datetime, timezone

db_url = URL.create(
    drivername='postgresql+psycopg2',
    username='indusense_user',
    password='ThEP@ssW0rd',
    host='localhost',
    port=5432,
    database='indusense_db',
)
engine = create_engine(db_url)
with engine.connect() as conn:
    print(conn.execute(text('SELECT version()')).scalar())


## 1. Lecture Bronze (parse_ok = True uniquement)

On ne traite que les lignes validées lors du TP4.


In [ ]:
df_tel = pd.read_sql(
    'SELECT * FROM bronze_telemetry WHERE parse_ok = true',
    engine
)
df_inc = pd.read_sql(
    'SELECT * FROM bronze_incidents WHERE parse_ok = true',
    engine
)
print(f'Telemetry valide : {len(df_tel):,} lignes')
print(f'Incidents valide : {len(df_inc):,} lignes')


## 2. Normalisation des machine_id

Les identifiants machine peuvent avoir des espaces, des casses différentes ou des préfixes hétérogènes.  
On applique une normalisation minimale : strip + uppercase.


In [ ]:
def normalize_machine_id(mid):
    if pd.isna(mid):
        return None
    return str(mid).strip().upper()

df_tel['machine_id_norm'] = df_tel['machine_id'].map(normalize_machine_id)
df_inc['machine_id_norm'] = df_inc['machine_id'].map(normalize_machine_id)

print('Telemetry — machines distinctes :', df_tel['machine_id_norm'].nunique())
print('Incidents — machines distinctes :', df_inc['machine_id_norm'].nunique())
communes = set(df_tel['machine_id_norm'].unique()) & set(df_inc['machine_id_norm'].unique())
print(f'Machines communes : {len(communes)}')
print(sorted(communes))


## 3. Parsing des timestamps

### 3a. Télémétrie — colonne `timestamp`


In [ ]:
df_tel['observed_at'] = pd.to_datetime(df_tel['timestamp'], errors='coerce', utc=True)
n_bad = df_tel['observed_at'].isna().sum()
print(f'Timestamps mal formés : {n_bad}')
print(df_tel[['timestamp', 'observed_at']].head(3))


### 3b. Incidents — colonnes `date` + `time`

In [ ]:
df_inc['occurred_at'] = pd.to_datetime(
    df_inc['date'].str.strip() + ' ' + df_inc['time'].str.strip(),
    errors='coerce'
).dt.tz_localize('UTC')
n_bad_inc = df_inc['occurred_at'].isna().sum()
print(f'Timestamps incidents mal formés : {n_bad_inc}')
print(df_inc[['date', 'time', 'occurred_at']].head(3))


## 4. Dépivotage des capteurs (format long)

On transforme les 5 colonnes capteurs en 5 lignes par relevé :
`sensor_type` contient le nom du capteur, `sensor_value` sa valeur, `unit` son unité.


In [ ]:
SENSOR_MAP = [
    ('temperature_c',     'temperature_c',     '°C'),
    ('pressure_bar',      'pressure_bar',      'bar'),
    ('voltage_mean_v',    'voltage_mean_v',    'V'),
    ('rotation_mean_rpm', 'rotation_mean_rpm', 'RPM'),
    ('pieces_produced',   'pieces_produced',   'pcs'),
]

chunks = []
for col, stype, unit in SENSOR_MAP:
    tmp = df_tel[['ingestion_batch_id', 'machine_id_norm', 'observed_at', col]].copy()
    tmp = tmp.rename(columns={'machine_id_norm': 'machine_id', col: 'sensor_value'})
    tmp['sensor_type'] = stype
    tmp['unit']        = unit
    chunks.append(tmp)

df_long = pd.concat(chunks, ignore_index=True)
df_long['sensor_value'] = pd.to_numeric(df_long['sensor_value'], errors='coerce')
print(f'Format long : {len(df_long):,} lignes ({len(df_tel):,} x {len(SENSOR_MAP)} capteurs)')
display(df_long.head(6))


## 5. Flags qualité Silver

### 5a. is_missing — valeur nulle

In [ ]:
df_long['is_missing'] = df_long['sensor_value'].isna()
print('Valeurs manquantes par capteur :')
print(df_long.groupby('sensor_type')['is_missing'].sum())


### 5b. is_duplicate — doublon (machine, observed_at, sensor_type)

Les doublons Bronze (`parse_ok=False`) ont été exclus en section 1.  
On vérifie qu'il n'en subsiste pas après normalisation des machine_id.


In [ ]:
dup_mask = df_long.duplicated(subset=['machine_id', 'observed_at', 'sensor_type'], keep='first')
df_long['is_duplicate'] = dup_mask
print(f'Doublons résiduels : {dup_mask.sum()}')


### 5c. is_outlier — méthode IQR par capteur

Une valeur est outlier si elle sort de l'intervalle `[Q1 - 1.5*IQR, Q3 + 1.5*IQR]`.


In [ ]:
def flag_outliers(group):
    v = group['sensor_value'].dropna()
    q1, q3 = v.quantile(0.25), v.quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return (group['sensor_value'] < lo) | (group['sensor_value'] > hi)

df_long['is_outlier'] = (
    df_long.groupby('sensor_type', group_keys=False)
           .apply(flag_outliers)
           .fillna(False)
)
print('Outliers par capteur :')
print(df_long.groupby('sensor_type')['is_outlier'].sum())


## 6. Batch Silver

In [ ]:
started_at = datetime.now(timezone.utc)

with engine.begin() as conn:
    batch_silver_tel = conn.execute(text("""
        INSERT INTO ingestion_batch (source_name, source_file, started_at, status)
        VALUES ('silver_telemetry', 'bronze_telemetry', :ts, 'running')
        RETURNING ingestion_batch_id
    """), {'ts': started_at}).scalar()

    batch_silver_inc = conn.execute(text("""
        INSERT INTO ingestion_batch (source_name, source_file, started_at, status)
        VALUES ('silver_incidents', 'bronze_incidents', :ts, 'running')
        RETURNING ingestion_batch_id
    """), {'ts': started_at}).scalar()

print(f'Batch silver_telemetry : {batch_silver_tel}')
print(f'Batch silver_incidents  : {batch_silver_inc}')


## 7. Ingestion silver_sensor_reading

In [ ]:
df_long['ingestion_batch_id'] = str(batch_silver_tel)

with engine.begin() as conn:
    conn.execute(text('TRUNCATE TABLE silver_sensor_reading RESTART IDENTITY'))

cols = ['ingestion_batch_id', 'machine_id', 'observed_at', 'sensor_type',
        'sensor_value', 'unit', 'is_missing', 'is_duplicate', 'is_outlier']
df_long[cols].to_sql('silver_sensor_reading', engine, if_exists='append',
                     index=False, method='multi', chunksize=2000)
print(f'silver_sensor_reading : {len(df_long):,} lignes insérées')


## 8. Silver Incidents

### 8a. Liaison avec la table operator

On récupère `operator_id` en faisant correspondre `operator_key` (nom normalisé).


In [ ]:
df_operators = pd.read_sql('SELECT operator_id, operator_key FROM operator', engine)

df_inc['operator_key'] = df_inc['operator_name'].str.strip().str.lower()
df_inc_silver = df_inc.merge(df_operators, on='operator_key', how='left')

n_no_op = df_inc_silver['operator_id'].isna().sum()
print(f'Incidents sans operator_id : {n_no_op}')


### 8b. is_label_event

Un incident est `is_label_event=True` si sa sévérité est >= 4 (incident critique pouvant servir de label supervisé).


In [ ]:
df_inc_silver['is_label_event'] = (
    pd.to_numeric(df_inc_silver['severity'], errors='coerce') >= 4
)
print(f'Label events : {df_inc_silver["is_label_event"].sum():,}')


### 8c. Ingestion silver_incident

In [ ]:
df_inc_silver['ingestion_batch_id'] = str(batch_silver_inc)
df_inc_silver['occurred_at']         = df_inc_silver['occurred_at']
df_inc_silver['incident_code']        = df_inc_silver['incident_id'].astype(str)
df_inc_silver['severity_int']         = pd.to_numeric(df_inc_silver['severity'], errors='coerce').astype('Int64')

with engine.begin() as conn:
    conn.execute(text('TRUNCATE TABLE silver_incident RESTART IDENTITY'))

cols_inc = ['ingestion_batch_id', 'incident_code', 'machine_id_norm',
            'operator_id', 'occurred_at', 'severity_int', 'shift', 'comment', 'is_label_event']
df_out = df_inc_silver[cols_inc].rename(columns={
    'machine_id_norm': 'machine_id',
    'severity_int':    'severity',
})
df_out.to_sql('silver_incident', engine, if_exists='append',
              index=False, method='multi', chunksize=500)
print(f'silver_incident : {len(df_out):,} lignes insérées')


## 9. Clôture des batches Silver

In [ ]:
finished_at = datetime.now(timezone.utc)

with engine.begin() as conn:
    conn.execute(text("""
        UPDATE ingestion_batch SET
            finished_at   = :fin,
            rows_read     = :read,
            rows_loaded   = :loaded,
            rows_rejected = :rejected,
            status        = 'done'
        WHERE ingestion_batch_id = :bid
    """), {'fin': finished_at,
           'read': len(df_long),
           'loaded': int((~df_long['is_missing'] & ~df_long['is_duplicate']).sum()),
           'rejected': int((df_long['is_missing'] | df_long['is_duplicate']).sum()),
           'bid': str(batch_silver_tel)})

    conn.execute(text("""
        UPDATE ingestion_batch SET
            finished_at   = :fin,
            rows_read     = :read,
            rows_loaded   = :loaded,
            rows_rejected = 0,
            status        = 'done'
        WHERE ingestion_batch_id = :bid
    """), {'fin': finished_at, 'read': len(df_inc), 'loaded': len(df_out),
           'bid': str(batch_silver_inc)})

print('Batches Silver clôturés')


## 10. Vérification finale

In [ ]:
display(pd.read_sql(text("""
    SELECT source_name, rows_read, rows_loaded, rows_rejected, status
    FROM ingestion_batch
    WHERE source_name LIKE 'silver%'
    ORDER BY started_at DESC
"""), engine))

print('--- silver_sensor_reading : stats outliers ---')
display(pd.read_sql(text("""
    SELECT sensor_type,
           COUNT(*)                             AS total,
           SUM(is_missing::int)                AS missing,
           SUM(is_duplicate::int)              AS duplicates,
           SUM(is_outlier::int)                AS outliers,
           ROUND(AVG(sensor_value)::numeric, 3) AS avg_value
    FROM silver_sensor_reading
    GROUP BY sensor_type
    ORDER BY sensor_type
"""), engine))

print('--- silver_incident : label events ---')
display(pd.read_sql(text("""
    SELECT is_label_event, COUNT(*) AS nb
    FROM silver_incident
    GROUP BY is_label_event
"""), engine))